# Lesson 03 - 智能体设计模式

在本课中，我们探讨构建高效 AI 智能体的三种基础设计模式：

1. **清晰的智能体指令** — 制定精确、定义角色的提示以引导智能体行为  
2. **使用 Pydantic 模型的结构化输出** — 确保智能体返回可预测、经过验证的数据  
3. **单一职责智能体** — 设计专注的智能体，每个智能体专注做好一件事

我们将将每种模式应用于一个**旅行目的地推荐系统**示例，逐步构建能够推荐目的地、检查可用性并处理后勤的系统。


## 设置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 模式 1：清晰的代理指令

最有效的模式也是最简单的：为你的代理编写清晰、详细的指令。

好的指令定义了：
- **代理是谁**（角色和语气）
- **代理应该做什么**（逐步职责）
- **代理应该如何表现**（约束和风格）

下面，我们创建一个具有明确指令的旅行礼宾代理，这些指令塑造了其每次回应的内容。


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## 模式 2：使用 Pydantic 模型的结构化输出

自由格式文本适合对话，但下游系统需要结构化数据。
通过将**Pydantic 模型**与**工具函数**结合使用，我们可以：

- 为代理的输出定义精确的模式
- 自动验证响应
- 可靠地将代理结果集成到应用逻辑中

我们还引入了一个返回目的地详细信息的工具，以便代理基于真实数据来支持其推荐。


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## 模式 3：单一职责代理

复杂任务通过将工作拆分给多个专注的代理来受益，每个代理只负责一项职责：

- 一个知道地点和可用性的**目的地专家**
- 一个处理航班、酒店和行程的**物流规划师**

这反映了软件工程中的*关注点分离*原则——每个代理更易于独立测试、维护和改进。


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

在本课中，我们将三种代理设计模式应用于旅游推荐场景：

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | 预先定义角色、职责和约束 | 一致、符合品牌的代理行为 |
| **Structured Output** | 使用 Pydantic 模型作为响应格式 | 验证的、机器可读的结果 |
| **Single Responsibility** | 给每个代理一个专注的任务 | 更易于测试、维护和组合 |

这些模式可以自然组合 —— 你可以在单一职责代理内结合明确指令和结构化输出，以构建稳健、生产准备的系统。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：  
本文档已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意自动翻译可能包含错误或不准确之处。原始文档的原文版本应被视为权威来源。对于重要信息，建议使用专业人工翻译。我们不对因使用本翻译而引起的任何误解或误释承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
